# Suzuki-m1 — treino de GPT do zero (português)

- Dataset: `Raivatv24/Suzuki-m1` (jsonl, `{"text": ...}`)
- Pipeline: baixa dataset → treina tokenizer BPE (se ainda nao tiver) → treina GPT do zero → checkpoint + push pro HF
- Aguarde o dataset estar no Hub antes de rodar (cheque `https://huggingface.co/Raivatv24/Suzuki-m1`)


In [ ]:
!pip install -q --upgrade transformers datasets tokenizers huggingface_hub accelerate

In [ ]:
import os
import json
import random
import math
import time
from itertools import islice

os.environ.setdefault("HF_HUB_DISABLE_XET", "1")  # evita a rota Xet do HF, que esta rejeitando pushes

import torch
import torch.nn as nn
from torch.optim import AdamW

from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    PreTrainedTokenizerFast,
    get_cosine_schedule_with_warmup,
)

DATA_REPO = "Raivatv24/Suzuki-m1"
MODEL_REPO = "Raivatv24/suzuki-m1-gpt"

from kaggle_secrets import UserSecretsClient
secret = UserSecretsClient()
HF_TOKEN = secret.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
api = HfApi(token=HF_TOKEN)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| gpu:", os.environ.get("KAGGLE_GPU_AVAILABLE_MEMORY", "?"))

In [ ]:
BLOCK = 256
N_LAYER = 6
N_HEAD = 6
N_EMBD = 384

VOCAB = 32000
BATCH = 16
ACCUM = 4
LR = 3e-4
WARMUP = 200
STEPS = 10000

SAVE_EVERY = 1000
EVAL_EVERY = 500
GEN_EVERY = 500
RESUME = True

TOKENIZER_DOCS = 80000
print("modelo ~35M params | block", BLOCK)

In [ ]:
train_ds = load_dataset(DATA_REPO, split="train", streaming=True)
val_ds = load_dataset(DATA_REPO, split="validation", streaming=True)
print("dataset ok")

In [ ]:
CACHE_DIR = "tokenizer_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
tk = None

if RESUME:
    try:
        api.hf_hub_download(repo_id=MODEL_REPO, filename="tokenizer.json",
                            local_dir=CACHE_DIR, repo_type="model")
    except Exception:
        pass

if os.path.exists(os.path.join(CACHE_DIR, "tokenizer.json")):
    tk = PreTrainedTokenizerFast(
        tokenizer_file=os.path.join(CACHE_DIR, "tokenizer.json"),
        eos_token="<|endoftext|>",
        pad_token="<|pad|>",
        unk_token="<|endoftext|>",
        bos_token="<|endoftext|>",
    )
    print("tokenizer reutilizado do Hub")

if tk is None:
    from tokenizers import (ByteLevelBPETokenizer, Tokenizer,
                            decoders, models, pre_tokenizers,
                            processors, trainers)
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    tok.post_processor = processors.ByteLevel(trim_offsets=False)
    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB,
        special_tokens=["<|endoftext|>", "<|pad|>"],
        unk_token="<|endoftext|>",
        min_frequency=2,
        initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    )
    def iter_docs(limit):
        c = 0
        for row in train_ds:
            yield row["text"]
            c += 1
            if c >= limit:
                return
    tok.train_from_iterator(iter_docs(TOKENIZER_DOCS), trainer=trainer)
    tok.save(os.path.join(CACHE_DIR, "tokenizer.json"))
    tk = PreTrainedTokenizerFast(
        tokenizer_file=os.path.join(CACHE_DIR, "tokenizer.json"),
        eos_token="<|endoftext|>",
        pad_token="<|pad|>",
        unk_token="<|endoftext|>",
        bos_token="<|endoftext|>",
    )
    print("tokenizer treinado do zero")
print("vocab:", tk.vocab_size, "| eos:", tk.eos_token_id, "| pad:", tk.pad_token_id)

In [ ]:
class BlockIter:
    def __init__(self, ds, tokenizer, block):
        self.ds = ds
        self.tk = tokenizer
        self.block = block

    def __iter__(self):
        buf = []
        pad = self.tk.pad_token_id
        eos = self.tk.eos_token_id
        for row in self.ds:
            ids = self.tk.encode(row["text"], add_special_tokens=False)
            if not ids:
                continue
            buf += ids + [eos]
            while len(buf) >= self.block:
                yield buf[:self.block]
                del buf[:self.block]
        if buf:
            buf += [pad] * (self.block - len(buf))
            yield buf

train_blocks = BlockIter(train_ds, tk, BLOCK)
val_blocks = BlockIter(val_ds, tk, BLOCK)
print("sampler ok")

In [ ]:
config = GPT2Config(
    vocab_size=tk.vocab_size,
    n_positions=BLOCK,
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    bos_token_id=tk.bos_token_id,
    eos_token_id=tk.eos_token_id,
    pad_token_id=tk.pad_token_id,
)
model = GPT2LMHeadModel(config)
nparam = sum(p.numel() for p in model.parameters())
print(f"{nparam/1e6:.1f}M parametros do zero")
model.to(device)

In [ ]:
opt = AdamW(model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.1)
sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=WARMUP, num_training_steps=STEPS)
criterion = nn.CrossEntropyLoss(ignore_index=tk.pad_token_id)

step = 0
ckpt_path = "ckpt.pt"

if RESUME:
    found = os.path.exists(ckpt_path)
    if not found:
        try:
            api.hf_hub_download(repo_id=MODEL_REPO, filename="ckpt.pt",
                                local_dir=".", repo_type="model")
            found = True
        except Exception:
            pass
    if found:
        ck = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        step = ck["step"]
        print("retomado do passo", step)

print("treino iniciando do passo", step)

In [ ]:
from torch.amp import GradScaler, autocast

scaler = GradScaler("cuda", enabled=(device == "cuda"))
train_iter = iter(train_blocks)
model.train()
t0 = time.time()

def evaluate():
    model.eval()
    with torch.no_grad():
        tot = 0.0
        n = 0
        for x in islice(val_blocks, 16):
            x = torch.tensor(x, dtype=torch.long, device=device).unsqueeze(0)
            with autocast("cuda", dtype=torch.float16, enabled=(device == "cuda")):
                out = model(input_ids=x, attention_mask=(x != tk.pad_token_id).long(), labels=x)
            tot += out.loss.item()
            n += 1
    model.train()
    return tot / max(n, 1)

def generate():
    model.eval()
    with torch.no_grad():
        x = torch.tensor([[tk.bos_token_id]], dtype=torch.long, device=device)
        out = model.generate(
            x,
            max_new_tokens=60,
            do_sample=True,
            top_k=50,
            temperature=0.8,
            pad_token_id=tk.pad_token_id,
            eos_token_id=tk.eos_token_id,
        )
    model.train()
    return tk.decode(out[0].tolist())

while step < STEPS:
    opt.zero_grad(set_to_none=True)
    xs = torch.stack(
        [torch.tensor(next(train_iter), dtype=torch.long, device=device)
         for _ in range(ACCUM)]
    )
    with autocast("cuda", dtype=torch.float16, enabled=(device == "cuda")):
        out = model(input_ids=xs, attention_mask=(xs != tk.pad_token_id).long(), labels=xs)
        loss = out.loss
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt)
    scaler.update()
    sched.step()
    step += 1

    if step % EVAL_EVERY == 0 or step % 100 == 0:
        el = (time.time() - t0) / 60
        print(f"step {step}/{STEPS} | loss {loss.item():.4f} | lr {sched.get_last_lr()[0]:.2e} | {el:.1f}min", flush=True)

    if step % EVAL_EVERY == 0:
        print("val loss:", round(evaluate(), 4))

    if step % GEN_EVERY == 0:
        print("--- amostra ---")
        print(generate()[:300])

    if step % SAVE_EVERY == 0:
        for tentativa in range(3):
            try:
                model.save_pretrained("model_save")
                tk.save_pretrained("model_save")
                api.create_repo(repo_id=MODEL_REPO, repo_type="model", exist_ok=True)
                api.upload_folder(folder_path="model_save", repo_id=MODEL_REPO,
                                  repo_type="model", commit_message=f"model step {step}")
                torch.save({"step": step, "model": model.state_dict(),
                            "opt": opt.state_dict(), "sched": sched.state_dict()}, ckpt_path)
                with open(ckpt_path, "rb") as f:
                    api.upload_file(path_or_fileobj=f, path_in_repo="ckpt.pt",
                                    repo_id=MODEL_REPO, repo_type="model",
                                    commit_message=f"ckpt step {step}")
                print("salvo+enviado step", step, flush=True)
                break
            except Exception as e:
                print(f"[aviso] falha no save step {step} (tentativa {tentativa + 1}): {e}",
                      flush=True)
                time.sleep(30)
                if tentativa == 2:
                    print("[aviso] desistindo do push deste step; contiinuando o treino", flush=True)

print("TREINO_CONCLUIDO")
generate()

## Como reutilizar

- Cada `SAVE_EVERY` o modelo e o checkpoint vão para `Raivatv24/suzuki-m1-gpt`.
- Se a sessão do Kaggle cair, rode de novo com `RESUME=True` — ele baixa `ckpt.pt` e continua de onde parou.
- Ao final, o modelo está em `Raivatv24/suzuki-m1-gpt` e pode ser carregado com:
```python
from transformers import pipeline
pl = pipeline("text-generation", model="Raivatv24/suzuki-m1-gpt")
pl("Era uma vez", max_new_tokens=60)
```
